# Tokenizer 实验（2.7）

本 Notebook 只实现作业 2.7 必做项：
- (a) TinyStories/OWT 各抽 10 篇文档，计算压缩率（bytes/token）
- (b) Tiny tokenizer 用在 OWT 样本时的效果
- (c) 吞吐量（bytes/s）与 Pile 825GB 时间估算
- (d) 编码 train/dev 并保存为 uint16 NumPy

不添加与作业无关功能。


In [4]:
from __future__ import annotations
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'cs336_basics').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cs336_basics.tokenizer.tokenizer import BPETokenizer

DATA_DIR = ROOT / 'dataset'
BPE_DIR = ROOT / 'artifacts' / 'bpe'
TOK_OUT = ROOT / 'artifacts' / 'tokenized'
TOK_OUT.mkdir(parents=True, exist_ok=True)

TINY_TRAIN = DATA_DIR / 'TinyStoriesV2-GPT4-train.txt'
TINY_VALID = DATA_DIR / 'TinyStoriesV2-GPT4-valid.txt'

TINY_BPE = BPE_DIR / 'tinystories_10k'

for p in [TINY_BPE/'vocab.json', TINY_BPE/'merges.txt']:
    assert p.exists(), f'Missing tokenizer file: {p} (先运行 bpe_experiment.ipynb)'

print('ROOT =', ROOT)


ROOT = /mnt/dataset0/gjt/CS336/assignment1-basics-main


In [6]:
def load_tokenizer(model_dir: Path):
    return BPETokenizer.from_files(
        vocab_filepath=model_dir/'vocab.json',
        merges_filepath=model_dir/'merges.txt',
        special_tokens=['<|endoftext|>'],
    )

def sample_docs(path: Path, n_docs=10, sep='<|endoftext|>', max_chars_per_doc=20000):
    text = path.read_text(encoding='utf-8', errors='ignore')
    docs = [d.strip() for d in text.split(sep) if d.strip()]
    return [d[:max_chars_per_doc] for d in docs[:n_docs]]

def bytes_per_token(tokenizer, docs):
    text = ''.join(docs)
    ids = tokenizer.encode(text)
    n_bytes = len(text.encode('utf-8'))
    return n_bytes / max(len(ids), 1), n_bytes, len(ids)

tiny_tok = load_tokenizer(TINY_BPE)
owt_tok = load_tokenizer(OWT_BPE)

tiny_docs_10 = sample_docs(TINY_TRAIN, n_docs=10)
owt_docs_10 = sample_docs(OWT_TRAIN, n_docs=10)
print('tiny_docs=', len(tiny_docs_10), 'owt_docs=', len(owt_docs_10))


ValueError: not enough values to unpack (expected 2, got 1)

## 2.7(a) 压缩率（bytes/token）


In [ ]:
rows = []
for corpus_name, docs in [('TinyStories-10docs', tiny_docs_10), ('OWT-10docs', owt_docs_10)]:
    for tok_name, tok in [('TinyTokenizer10K', tiny_tok), ('OWTTokenizer32K', owt_tok)]:
        bpt, n_bytes, n_tok = bytes_per_token(tok, docs)
        rows.append({'corpus': corpus_name, 'tokenizer': tok_name, 'bytes': n_bytes, 'tokens': n_tok, 'bytes_per_token': bpt})

df_comp = pd.DataFrame(rows).sort_values(['corpus', 'tokenizer']).reset_index(drop=True)
df_comp


## 2.7(b) Tiny tokenizer 在 OWT 上的表现


In [ ]:
owt_on_tiny = float(df_comp.query("corpus=='OWT-10docs' and tokenizer=='TinyTokenizer10K'")['bytes_per_token'])
owt_on_owt  = float(df_comp.query("corpus=='OWT-10docs' and tokenizer=='OWTTokenizer32K'")['bytes_per_token'])
ratio = owt_on_tiny / owt_on_owt

print('OWT with Tiny tokenizer bytes/token:', round(owt_on_tiny, 4))
print('OWT with OWT tokenizer  bytes/token:', round(owt_on_owt, 4))
print('ratio Tiny/OWT:', round(ratio, 4))


## 2.7(c) 吞吐量与 Pile 时间估算


In [ ]:
def measure_throughput_bps(tokenizer, path: Path, repeats=2):
    n_bytes = path.stat().st_size
    best = float('inf')
    for _ in range(repeats):
        t0 = time.perf_counter()
        with path.open('r', encoding='utf-8', errors='ignore') as f:
            _ = sum(1 for _id in tokenizer.encode_iterable(f))
        best = min(best, time.perf_counter() - t0)
    return n_bytes / best

tiny_probe = TINY_VALID if TINY_VALID.exists() else TINY_TRAIN
owt_probe  = OWT_VALID if OWT_VALID.exists() else OWT_TRAIN

tp_tiny = measure_throughput_bps(tiny_tok, tiny_probe, repeats=2)
tp_owt  = measure_throughput_bps(owt_tok, owt_probe, repeats=2)
PILE_BYTES = 825 * (1024**3)

pile_tiny_h = PILE_BYTES / tp_tiny / 3600
pile_owt_h  = PILE_BYTES / tp_owt / 3600

print('Tiny throughput MB/s:', round(tp_tiny/(1024**2), 2))
print('OWT throughput  MB/s:', round(tp_owt/(1024**2), 2))
print('Pile estimate (Tiny tokenizer, hours):', round(pile_tiny_h, 2))
print('Pile estimate (OWT tokenizer,  hours):', round(pile_owt_h, 2))


## 2.7(d) 编码 train/dev 为 uint16 `.npy`

说明：默认不自动执行全量编码，避免误触发长任务。将 `RUN_FULL_ENCODING=True` 后运行本单元。


In [ ]:
def count_tokens(tokenizer, path: Path) -> int:
    c = 0
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for _id in tokenizer.encode_iterable(f):
            c += 1
    return c

def encode_file_to_uint16_npy(tokenizer, src: Path, dst: Path):
    vmax = max(tokenizer.vocab.keys())
    assert vmax < 65536, f'max token id {vmax} exceeds uint16'

    n = count_tokens(tokenizer, src)
    arr = np.lib.format.open_memmap(dst, mode='w+', dtype=np.uint16, shape=(n,))

    i = 0
    with src.open('r', encoding='utf-8', errors='ignore') as f:
        for tid in tokenizer.encode_iterable(f):
            arr[i] = tid
            i += 1
    arr.flush()
    return n

RUN_FULL_ENCODING = False
if RUN_FULL_ENCODING:
    plan = [
        ('tiny_train', tiny_tok, TINY_TRAIN, TOK_OUT/'tiny_train_uint16.npy'),
        ('tiny_valid', tiny_tok, TINY_VALID, TOK_OUT/'tiny_valid_uint16.npy'),
        ('owt_train', owt_tok, OWT_TRAIN, TOK_OUT/'owt_train_uint16.npy'),
        ('owt_valid', owt_tok, OWT_VALID, TOK_OUT/'owt_valid_uint16.npy'),
    ]
    for name, tok, src, dst in plan:
        if not src.exists():
            print('[skip]', name, 'missing', src)
            continue
        t0 = time.perf_counter()
        n = encode_file_to_uint16_npy(tok, src, dst)
        print('[done]', name, 'tokens=', n, 'time=', round(time.perf_counter()-t0, 1), 's', 'file=', dst)
else:
    print('RUN_FULL_ENCODING=False, skipped full encoding.')


## 可直接提交的 2.7 答案草稿（1-2 句）


In [ ]:
tiny_on_tiny = float(df_comp.query("corpus=='TinyStories-10docs' and tokenizer=='TinyTokenizer10K'")['bytes_per_token'])

ans_a = (
    f'在 10 篇 TinyStories 和 10 篇 OWT 样本上，两个分词器的 bytes/token 如表所示；'
    f'其中 Tiny→Tiny={tiny_on_tiny:.3f}, OWT→OWT={owt_on_owt:.3f}。'
)
ans_b = (
    f'用 Tiny 分词器处理 OWT 样本时 bytes/token={owt_on_tiny:.3f}，而 OWT 分词器为 {owt_on_owt:.3f}（比值 {ratio:.3f}）；'
    '说明 Tiny 词表跨到网页域时压缩通常变差。'
)
ans_c = (
    f'实测吞吐量约 Tiny={tp_tiny/(1024**2):.2f}MB/s, OWT={tp_owt/(1024**2):.2f}MB/s；'
    f'估算分词 825GB Pile 约需 Tiny={pile_tiny_h:.1f}h, OWT={pile_owt_h:.1f}h。'
)
ans_d = (
    '已提供将 train/dev 序列化为 uint16 `.npy` 的代码；'
    'uint16 适合因为词表 id < 65536，每个 token 仅 2 字节，能显著节省存储与 I/O。'
)

print('2.7(a):', ans_a)
print('
2.7(b):', ans_b)
print('
2.7(c):', ans_c)
print('
2.7(d):', ans_d)
